In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RadarComparison"
# dataType = "RadarComparison_Interpolation"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Load Model Directory Class
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_TRACER = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_Hawaii = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_PRECIP = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
##################
#FUNCTIONS

In [ ]:
def MakeLatLonGrids(ModelData):
    
    longitudeGrid, latitudeGrid = np.meshgrid(
        ModelData.longitude,
        ModelData.latitude,
        indexing="ij"
    )
    
    longitudeGrid = xr.DataArray(
        longitudeGrid,
        dims=("longitude", "latitude"),
        coords={
            "longitude": ModelData.longitude,
            "latitude": ModelData.latitude,
        },
        name="longitude"
    )
    
    latitudeGrid = xr.DataArray(
        latitudeGrid,
        dims=("longitude", "latitude"),
        coords={
            "longitude": ModelData.longitude,
            "latitude": ModelData.latitude,
        },
        name="latitude"
    )
    
    return longitudeGrid,latitudeGrid

In [ ]:
def Calculate_XY_FromRadar(RadarLocation,
                          longitudeGrid, latitudeGrid):
    """
    Calculates local Cartesian coordinates (x, y) relative to a radar location
    using a tangent-plane approximation.

    x = R * cos(Lat0) * (Lon - Lon0)
    y = R * (Lat - Lat0)

    """

    Lat0, Lon0 = RadarLocation
    EarthRadius = 6371.0  # km

    # Convert to radians
    Lat0Rad = np.deg2rad(Lat0)

    dLon = np.deg2rad(longitudeGrid - Lon0)
    dLat = np.deg2rad(latitudeGrid  - Lat0)

    # Local Cartesian coordinates (km)
    xGrid = EarthRadius * np.cos(Lat0Rad) * dLon
    yGrid = EarthRadius * dLat

    xGrid.name = "x_from_radar"
    yGrid.name = "y_from_radar"

    xGrid.attrs["units"] = "km"
    yGrid.attrs["units"] = "km"

    return xGrid, yGrid


In [ ]:
def CalculateRange(x,y):
    r = x**2 + y**2
    r = np.sqrt(r)
    return r

In [ ]:
## PlotData

COASTLINE_FEATURE = cfeature.COASTLINE
BORDERS_FEATURE   = cfeature.BORDERS
STATES_FEATURE    = cfeature.STATES
# def PlotData(dataArray, levels=15, cmap="viridis"):

#     fig = plt.figure(figsize=(8, 6))
#     ax = plt.axes(projection=ccrs.PlateCarree())

#     cf = ax.contourf(
#         dataArray.longitude,
#         dataArray.latitude,
#         dataArray,#.transpose("latitude", "longitude"),
#         levels=levels,
#         cmap=cmap,
#         transform=ccrs.PlateCarree()
#     )

#     plt.colorbar(cf, ax=ax, label="Range (km)")
#     ax.coastlines()
#     ax.add_feature(cfeature.BORDERS, edgecolor="white",facecolor="none",)
#     ax.add_feature(cfeature.STATES, edgecolor="white",facecolor="none",)

#     plt.tight_layout()
#     return fig, ax

import cartopy.mpl.ticker as cticker

def PlotData(dataArray, levels=15, cmap="viridis"):

    fig = plt.figure(figsize=(8, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())

    cf = ax.contourf(
        dataArray.longitude,
        dataArray.latitude,
        dataArray.T,  # assumes dims (latitude, longitude) OR broadcasting works
        levels=levels,
        cmap=cmap,
        transform=ccrs.PlateCarree()
    )

    plt.colorbar(cf, ax=ax)

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, edgecolor="white", facecolor="none")
    ax.add_feature(cfeature.STATES,  edgecolor="white", facecolor="none")

    # --- Gridlines with labels ---
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.0,   # hide grid lines
        color="none",    # hide grid lines
        alpha=0.0
    )
    
    gl.top_labels = False
    gl.right_labels = False
    
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    plt.tight_layout()
    return fig, ax



In [ ]:
##################
#CALCULATING

In [ ]:
longitudeGrid.plot()

In [ ]:
#*#*#*
#TRACER/Hawaii has multiple radars, need to figure out radius


RadarLocation_Hawaii_1 = (22, -160) #*EXAMPLE
RadarLocation_Hawaii_2 = (22, -158) #*EXAMPLE
RadarLocation_Hawaii_3 = (22, -156) #*EXAMPLE
[longitudeGrid,latitudeGrid] = MakeLatLonGrids(ModelData_Hawaii)
[xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation_Hawaii_1,
                          longitudeGrid, latitudeGrid)
rGrid_1 = CalculateRange(xGrid,yGrid)

[xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation_Hawaii_2,
                          longitudeGrid, latitudeGrid)
rGrid_2 = CalculateRange(xGrid,yGrid)

[xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation_Hawaii_3,
                          longitudeGrid, latitudeGrid)
rGrid_3 = CalculateRange(xGrid,yGrid)








rGrid_combined = xr.concat([rGrid_1, rGrid_2, rGrid_3], 
                      dim="radar").min(dim="radar", skipna=True)






PlotData(rGrid_1)
PlotData(rGrid_2)
PlotData(rGrid_3)


Noise_1km = -44.4
SNR_constant = (Noise_1km+20*np.log10(rGrid_combined))

PlotData(SNR_constant,cmap='RdBu')


In [ ]:
RadarLocation_PRECIP = (24.82, 120.91)
[longitudeGrid,latitudeGrid] = MakeLatLonGrids(ModelData_PRECIP)
[xGrid, yGrid] = Calculate_XY_FromRadar(RadarLocation_PRECIP,
                          longitudeGrid, latitudeGrid)
rGrid = CalculateRange(xGrid,yGrid)

In [ ]:
PlotData(rGrid)
plt.scatter(120.91,24.82, color='red')

In [ ]:
# SNR (Signal to noise ratio) = Z_hh - (Noise_1km + 20*log10(r)) > 0
# ==> Z_hh > (Noise_1km + 20*log10(r))

Noise_1km = -44.4
SNR_constant = (Noise_1km+20*np.log10(rGrid))

PlotData(SNR_constant,cmap='RdBu')

In [ ]:
refl1 = ModelData_PRECIP.GetDataTimestep_diag(t=150,varName="refl10cm",
                                             printout=False)
refl2 = refl1.where(refl1 > SNR_constant)

In [ ]:
#INSPECT ALGORITHM RESULTS AT SIINGLE LEVEL

In [ ]:
PlotData(refl1.isel(nVertLevels=10),cmap='turbo')

In [ ]:
PlotData(refl2.isel(nVertLevels=10),cmap='turbo')

In [ ]:
#INSPECT ALGORITHM RESULTS AT All LEVEL

In [ ]:
a = refl1.mean(dim="latitude")
b = refl2.mean(dim="latitude")

xr.concat(
    [a, b],
    dim=xr.DataArray(
        ["refl1", "refl2"],
        dims="panel",
        name="panel"
    )
).plot(
    col="panel",
    col_wrap=1,
    cmap="turbo",
    figsize=(6, 6)
)

In [ ]:
a = refl1.mean(dim="longitude")
b = refl2.mean(dim="longitude")

xr.concat(
    [a, b],
    dim=xr.DataArray(
        ["refl1", "refl2"],
        dims="panel",
        name="panel"
    )
).plot(
    col="panel",
    col_wrap=1,
    cmap="turbo",
    figsize=(6, 6)
)